# Machine Learning and MLflow Examples

This notebook introduces common Databricks machine learning workflow patterns:

- feature engineering with Spark
- train and test data preparation
- MLflow experiment tracking
- model training and evaluation
- model registration workflow concepts

## Setup

This example uses a small in-memory dataset for demonstration. In production, features would typically come from curated Delta tables in Unity Catalog.

In [ ]:
import mlflow
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
feature_data = [
    (1, 10, 2, 300.0),
    (2, 15, 3, 450.0),
    (3, 8, 1, 210.0),
    (4, 20, 4, 600.0),
    (5, 12, 2, 360.0),
    (6, 18, 3, 540.0),
    (7, 9, 1, 250.0),
    (8, 22, 5, 710.0)
]

raw_df = spark.createDataFrame(feature_data, ["customer_id", "sessions", "orders", "revenue"])
display(raw_df)

## Feature engineering

Feature engineering in Databricks often starts with Spark transformations over curated data.

In [ ]:
features_df = (
    raw_df
    .withColumn("avg_order_value", F.col("revenue") / F.col("orders"))
    .withColumn("session_intensity", F.col("sessions") / F.col("orders"))
)

display(features_df)

## Assemble features and split data

Spark ML models expect features in a vector column.

In [ ]:
assembler = VectorAssembler(
    inputCols=["sessions", "orders", "avg_order_value", "session_intensity"],
    outputCol="features"
)

model_df = assembler.transform(features_df).select("features", F.col("revenue").alias("label"))
train_df, test_df = model_df.randomSplit([0.75, 0.25], seed=42)

print({"train_count": train_df.count(), "test_count": test_df.count()})

## Track a training run with MLflow

MLflow is commonly used in Databricks to track model parameters, metrics, artifacts, and versions.

In [ ]:
experiment_name = "/Shared/mlflow-demo-revenue-prediction"
mlflow.set_experiment(experiment_name)

lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=10, regParam=0.1)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")

with mlflow.start_run(run_name="linear-regression-demo"):
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("regParam", 0.1)

    model = lr.fit(train_df)
    predictions_df = model.transform(test_df)
    rmse = evaluator.evaluate(predictions_df)

    mlflow.log_metric("rmse", rmse)

    print({"rmse": rmse})

## Inspect predictions

Model quality should always be checked against actual outputs, not only logged as a single metric.

In [ ]:
display(predictions_df.select("label", "prediction"))

## Model registration concept

In production, the next step after evaluation is often registering a model version and promoting it through validation stages.

In [ ]:
model_name = "demo_revenue_prediction_model"
print(f"Next production step: register a model like {model_name} in your MLflow model registry or governed catalog-backed registry.")

## Practical guidance

- Build features from curated Delta tables
- Track every meaningful training run with MLflow
- Log both parameters and metrics
- Evaluate models with real business-relevant checks, not only one score
- Separate experimentation from production promotion and deployment